# Solutions · 05 · Discrete-time Markov chains

Worked solutions to the exercises of [notebook 05](../notebooks/05_discrete_time_markov_chains.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engstoch is missing): install it from GitHub
    import engstoch
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engstoch"], check=True)
import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engstoch import markov as mk, models, datasets
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

R = np.random.default_rng      # R(seed): a fresh, reproducible generator

## Exercise 1

A machine is good, worn or broken; write a 3-state chain with a repair policy of your choice, find its stationary law and the long-run fraction of broken days, and check by simulating 10^5 days.

In [2]:
P = np.array([[0.90, 0.08, 0.02],      # good: wears or breaks
              [0.00, 0.85, 0.15],      # worn: preventive repair only when broken in this policy
              [0.70, 0.00, 0.30]])     # broken: repaired to good with prob. 0.7 per day
c = mk.MarkovChain(P, ["good", "worn", "broken"])
pi = c.stationary()
path = c.simulate(100_000, "good", R(1))
print("stationary law:", pi.round(4).tolist(), "\nsimulated fractions:", (np.bincount(path, minlength=3) / len(path)).round(4).tolist())
P2 = P.copy(); P2[1] = [0.5, 0.45, 0.05]                    # alternative: repair worn machines (half of them per day)
print("with preventive repair of worn machines: broken", mk.MarkovChain(P2).stationary()[2].round(4), "vs", pi[2].round(4))

stationary law: [0.5966, 0.3182, 0.0852] 
simulated fractions: [0.6009, 0.3135, 0.0856]
with preventive repair of worn machines: broken 0.0329 vs 0.0852


Under this policy the machine is broken about 8.5 % of days; the simulation agrees. Changing one row - repairing
worn machines before they fail - cuts broken days by more than half. The stationary law turns a maintenance
policy into a number that can be costed.

## Exercise 2

Fit a separate two-state chain to each season of `weather.csv`, then simulate ten years from the seasonal model and from the homogeneous model. Compare the distribution of the longest wet spell per year with the data.

In [3]:
wd = pd.read_csv(datasets.path("weather.csv"), comment="#")
x = wd.wet.to_numpy(); doy = wd.day_of_year.to_numpy()
q = (doy - 1) // 91 % 4
P_s = []
for s in range(4):
    idx = np.flatnonzero(q[:-1] == s)
    c = np.zeros((2, 2)); np.add.at(c, (x[idx], x[idx + 1]), 1)
    P_s.append(c / c.sum(axis=1, keepdims=True))
P_h = mk.estimate(x, 2)["P"]
def simulate(Pfun, g, years=10):
    y = np.zeros(365 * years, dtype=int)
    for k in range(1, len(y)):
        y[k] = g.random() < Pfun(k)[y[k - 1], 1]
    return y
def longest(y):
    return [max((len(r) for r in np.split(yy, np.flatnonzero(np.diff(yy)) + 1) if r[0] == 1), default=0) for yy in y.reshape(-1, 365)]
obs = longest(x)
sea = sum((longest(simulate(lambda k: P_s[(k % 365) // 91 % 4], R(10 + i))) for i in range(10)), [])
hom = sum((longest(simulate(lambda k: P_h, R(30 + i))) for i in range(10)), [])
print(f"longest wet spell per year: data mean {np.mean(obs):.2f}, seasonal model {np.mean(sea):.2f}, homogeneous model {np.mean(hom):.2f}")
print(f"  upper quartile: data {np.quantile(obs, 0.75):.1f}, seasonal {np.quantile(sea, 0.75):.1f}, homogeneous {np.quantile(hom, 0.75):.1f}")

longest wet spell per year: data mean 7.80, seasonal model 6.63, homogeneous model 6.72
  upper quartile: data 9.0, seasonal 8.0, homogeneous 8.0


Neither chain reproduces the annual maximum: both give about 6.7 days against 7.8 in the data. The quarterly
chain is no better than the homogeneous one because averaging over a quarter still hides the peak of the wet
season - the generating probabilities vary smoothly through the year - and annual maxima are driven by exactly
that peak. (With ten years the data's mean has a standard error of about 0.6 days, so the gap is real.) The
next step is a chain whose transition probabilities vary smoothly with the day of the year, fitted by logistic
regression of tomorrow's state on today's state and a seasonal cosine.

## Exercise 3

**Implement it yourself:** write the MLE of P with standard errors from a path of state indices, and check it against `markov.estimate` on the weather data.

In [4]:
def mle(path, m):
    counts = np.zeros((m, m)); np.add.at(counts, (path[:-1], path[1:]), 1)
    n_i = counts.sum(axis=1, keepdims=True)
    P = counts / n_i
    return P, np.sqrt(P * (1 - P) / n_i)
x = pd.read_csv(datasets.path("weather.csv"), comment="#").wet.to_numpy()
P, se = mle(x, 2)
lib = mk.estimate(x, 2)
print("mine:\n", P.round(5), "\n", se.round(5), "\nlibrary agrees:", np.allclose(P, lib["P"]) and np.allclose(se, lib["std_error"]))

mine:
 [[0.80872 0.19128]
 [0.52947 0.47053]] 
 [[0.00759 0.00759]
 [0.01605 0.01605]] 
library agrees: True


Given the number of departures n_i from state i, the next states are multinomial, so the MLE is the observed
fraction and its standard error the binomial one. The counts matrix is all the data the first-order chain uses.